# 实验
对比yolo26n和集成模型mAp 推理速度

## 导入数据集并切分

## 使用 Ultralytics 云端数据集

本数据集已上传到 Ultralytics Platform：

- 数据集 URL: https://platform.ultralytics.com/yan-congjie/datasets/driverless-vehicle
- 训练时直接使用云端引用: ul://yan-congjie/datasets/driverless-vehicle
- Ultralytics 会自动解析并流式拉取数据，无需手动下载、解压或划分
- 类别：
  - 0: UGV (无人车)
  - 1: USV (无人艇)


In [1]:
# 确认云端数据集引用格式
DATA = "ul://yan-congjie/datasets/driverless-vehicle"

print("使用 Ultralytics 云端数据集:")
print("DATA:", DATA)
print()
print("训练时 Ultralytics 会自动:")
print("- 验证数据集访问权限")
print("- 流式下载训练和验证数据")
print("- 解析类别和图像路径")
print("- 无需本地解压或手动划分")


使用 Ultralytics 云端数据集:
DATA: ul://yan-congjie/datasets/driverless-vehicle

训练时 Ultralytics 会自动:
- 验证数据集访问权限
- 流式下载训练和验证数据
- 解析类别和图像路径
- 无需本地解压或手动划分


更新使用方式：
1. 从 Platform 下载 driverless-vehicle.ndjson
2. 上传到 Colab：/content/driverless-vehicle.ndjson
3. 运行 Notebook，Platform 会自动：
   - 下载数据集到缓存
   - 处理 train/val/test 划分
   - 管理类别映射
不需要手动下载 ZIP、解压或创建目录结构。

## 导入yolo26n模型，集成edgevit

In [2]:
# Colab 新运行时初始化：拉取 YOLO26 和官方 EdgeViT
from pathlib import Path
import subprocess

CONTENT = Path("/content")
ULTRALYTICS = CONTENT / "ultralytics"
EDGEVIT = CONTENT / "edgevit"

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, cwd=cwd, shell=True, check=True)

if not (ULTRALYTICS / ".git").exists():
    run("git clone --depth 1 https://github.com/ultralytics/ultralytics.git /content/ultralytics")
else:
    print("已有 /content/ultralytics，跳过 clone")

if not (EDGEVIT / ".git").exists():
    run("git clone --depth 1 https://github.com/saic-fi/edgevit.git /content/edgevit")
else:
    print("已有 /content/edgevit，跳过 clone")

run("git fetch --depth 1 origin main", cwd=ULTRALYTICS)
run("git checkout --detach origin/main", cwd=ULTRALYTICS)

assert (ULTRALYTICS / "ultralytics" / "cfg" / "models" / "26" / "yolo26.yaml").exists()
assert (EDGEVIT / "src" / "edgevit.py").exists()

run("python -m pip install -q -e /content/ultralytics --no-deps")

print("YOLO26 commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ULTRALYTICS, text=True
).strip())
print("EdgeViT commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=EDGEVIT, text=True
).strip())


已有 /content/ultralytics，跳过 clone
已有 /content/edgevit，跳过 clone
$ git fetch --depth 1 origin main
$ git checkout --detach origin/main
$ python -m pip install -q -e /content/ultralytics --no-deps
YOLO26 commit: d0cfceaf16f8d13dc3e62270744e8f8a44640448
EdgeViT commit: df3d070addc3dc761173429426eab5edeb0d6343


In [3]:
# 建立 YOLO26 基线，确认 Colab、PyTorch、CUDA 和模型加载正常
%cd /content/ultralytics

import sys

# Colab 输出流可能使用 GBK；Ultralytics 日志包含 Unicode 符号。
# 统一为 UTF-8，避免 baseline 加载时触发 UnicodeEncodeError。
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

repo = "/content/ultralytics"
if repo not in sys.path:
    sys.path.insert(0, repo)

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Ultralytics:", ultralytics.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

baseline = YOLO("yolo26n.pt")
print("YOLO26 baseline loaded")


/content/ultralytics
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.4.6
Ultralytics: /content/ultralytics/ultralytics/__init__.py
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
YOLO26 baseline loaded


In [4]:
# 安装 timm，并记录当前版本。
# EdgeViT 不依赖 timm 中的 EdgeViT 注册名，但官方源码仍使用 timm 的基础层。
%pip install -q timm

import timm

print("timm:", timm.__version__)
print("timm EdgeViT entries:", timm.list_models("*edgevit*"))


timm: 1.0.31
timm EdgeViT entries: ['edgevit_s', 'edgevit_xs', 'edgevit_xxs']


In [5]:
# 兼容官方 EdgeViT 源码与新版 timm 的导入路径
from pathlib import Path
import shutil

edgevit_source = Path("/content/edgevit/src/edgevit.py")
source_text = edgevit_source.read_text(encoding="utf-8")

old_imports = '''from timm.models.vision_transformer import _cfg
from timm.models.registry import register_model
from timm.models.layers import trunc_normal_, DropPath, to_2tuple
'''

new_imports = '''from timm.models.vision_transformer import _cfg

try:
    from timm.models.registry import register_model
except ImportError:
    from timm.models._registry import register_model

try:
    from timm.models.layers import trunc_normal_, DropPath, to_2tuple
except ImportError:
    from timm.layers import trunc_normal_, DropPath, to_2tuple
'''

backup = Path(str(edgevit_source) + ".orig")
if not backup.exists():
    shutil.copy2(edgevit_source, backup)

if old_imports in source_text:
    source_text = source_text.replace(old_imports, new_imports, 1)
elif "from timm.models._registry import register_model" in source_text:
    print("EdgeViT 导入兼容补丁已经存在")
else:
    raise RuntimeError("没有找到预期的 EdgeViT timm 导入块，停止以避免误改源码")

edgevit_source.write_text(source_text, encoding="utf-8")
print("patched:", edgevit_source)


EdgeViT 导入兼容补丁已经存在
patched: /content/edgevit/src/edgevit.py


In [6]:
# 创建 EdgeViT -> YOLO26 的多尺度骨干适配器
from pathlib import Path

root = Path("/content/ultralytics")
extra = root / "ultralytics" / "nn" / "modules" / "extramodules"
extra.mkdir(parents=True, exist_ok=True)
(extra / "__init__.py").touch()

adapter = r'''# Ultralytics AGPL-3.0 License - https://ultralytics.com/license
# EdgeViT adapter for YOLO26.
from __future__ import annotations

import sys
from functools import partial

import torch
from torch import nn
import torch.nn.functional as F

EDGEVIT_SRC = "/content/edgevit/src"
if EDGEVIT_SRC not in sys.path:
    sys.path.insert(0, EDGEVIT_SRC)

from edgevit import EdgeVit


_VARIANTS = {
    "xxs": {
        "depth": [1, 1, 3, 2],
        "embed_dim": [36, 72, 144, 288],
        "head_dim": 36,
        "sr_ratios": [4, 2, 2, 1],
    },
    "xs": {
        "depth": [1, 1, 3, 1],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
    "s": {
        "depth": [1, 2, 5, 3],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
}


class _EdgeViTBackbone(nn.Module):
    # 输出 EdgeViT 的 stride 8/16/32 特征 [P3, P4, P5]。

    def __init__(self, variant="xxs", img_size=640):
        super().__init__()

        cfg = _VARIANTS[variant]
        self.img_size = img_size
        self.channel = list(cfg["embed_dim"][1:])

        self.model = EdgeVit(
            img_size=img_size,
            num_classes=0,
            mlp_ratio=[4] * 4,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-6),
            **cfg,
        )

    def forward(self, x: torch.Tensor):
        original_h, original_w = x.shape[-2:]

        # 官方 PatchEmbed 会检查输入尺寸。YOLO 构建 stride 时可能传入较小张量，
        # 所以统一到 640×640，完成骨干计算后再缩放回当前输入的 P3/P4/P5 尺寸。
        if (original_h, original_w) != (self.img_size, self.img_size):
            x = F.interpolate(
                x,
                size=(self.img_size, self.img_size),
                mode="bilinear",
                align_corners=False,
            )

        # Stage 1: stride 4
        x = self.model.patch_embed1(x)
        x = self.model.pos_drop(x)
        for block in self.model.blocks1:
            x = block(x)

        # Stage 2: stride 8，作为 P3
        x = self.model.patch_embed2(x)
        for block in self.model.blocks2:
            x = block(x)
        p3 = x

        # Stage 3: stride 16，作为 P4
        x = self.model.patch_embed3(x)
        for block in self.model.blocks3:
            x = block(x)
        p4 = x

        # Stage 4: stride 32，作为 P5
        x = self.model.patch_embed4(x)
        for block in self.model.blocks4:
            x = block(x)
        p5 = self.model.norm(x)

        # 让输出尺寸始终与 YOLO 当前输入对应。
        p3 = F.interpolate(
            p3,
            size=(max(original_h // 8, 1), max(original_w // 8, 1)),
            mode="nearest",
        )
        p4 = F.interpolate(
            p4,
            size=(max(original_h // 16, 1), max(original_w // 16, 1)),
            mode="nearest",
        )
        p5 = F.interpolate(
            p5,
            size=(max(original_h // 32, 1), max(original_w // 32, 1)),
            mode="nearest",
        )

        return [p3, p4, p5]


class EdgeViT_XXS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xxs")


class EdgeViT_XS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xs")


class EdgeViT_S(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("s")
'''

adapter_path = extra / "edgevit.py"
adapter_path.write_text(adapter, encoding="utf-8")
print("adapter:", adapter_path)


adapter: /content/ultralytics/ultralytics/nn/modules/extramodules/edgevit.py


In [7]:
# 独立验证 EdgeViT 适配器的三个输出尺度
import importlib.util
import torch

adapter_path = "/content/ultralytics/ultralytics/nn/modules/extramodules/edgevit.py"
spec = importlib.util.spec_from_file_location("edgevit_adapter", adapter_path)
edgevit_adapter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(edgevit_adapter)

assert issubclass(edgevit_adapter.EdgeViT_XXS, torch.nn.Module)

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = edgevit_adapter.EdgeViT_XXS().to(device).eval()
x = torch.randn(1, 3, 640, 640, device=device)

with torch.inference_mode():
    outputs = backbone(x)

print("channels:", backbone.channel)
for index, output in enumerate(outputs):
    print(index, tuple(output.shape))

assert backbone.channel == [72, 144, 288]
assert [tuple(output.shape[-2:]) for output in outputs] == [
    (80, 80),
    (40, 40),
    (20, 20),
]
print("EdgeViT adapter test passed")


channels: [72, 144, 288]
0 (1, 72, 80, 80)
1 (1, 144, 40, 40)
2 (1, 288, 20, 20)
EdgeViT adapter test passed


### 修改 Ultralytics 模块注册和 parse_model

In [7]:
# 注册 EdgeViT，并对 Ultralytics 的 parse_model 做最小修改
from pathlib import Path
import shutil

root = Path("/content/ultralytics")
modules_init = root / "ultralytics" / "nn" / "modules" / "___init__.py"
modules_init = root / "ultralytics" / "nn" / "modules" / "__init__.py"
tasks_path = root / "ultralytics" / "nn" / "tasks.py"

registration = """
from .extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
"""

modules_text = modules_init.read_text(encoding="utf-8")
if "from .extramodules.edgevit import" not in modules_text:
    modules_backup = Path(str(modules_init) + ".bak")
    if not modules_backup.exists():
        shutil.copy2(modules_init, modules_backup)
    modules_init.write_text(modules_text + registration, encoding="utf-8")
    print("已注册 EdgeViT 到 modules/__init__.py")
else:
    print("modules/__init__.py 已经注册 EdgeViT")

tasks_text = tasks_path.read_text(encoding="utf-8")

if "from ultralytics.nn.modules.extramodules.edgevit import" not in tasks_text:
    import_marker = "from ultralytics.nn.autobackend import check_class_names"
    imports = """from ultralytics.nn.modules.extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
"""
    if import_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到预期的导入标记，未执行修改")
    tasks_text = tasks_text.replace(import_marker, imports + "\n" + import_marker, 1)

if "edgevit_modules = frozenset" not in tasks_text:
    loop_marker = '    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):  # from, number, module, args'
    collection = """    edgevit_modules = frozenset({
        EdgeViT_XXS,
        EdgeViT_XS,
        EdgeViT_S,
    })
"""
    if loop_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 parse_model 循环，未执行修改")
    tasks_text = tasks_text.replace(loop_marker, collection + loop_marker, 1)

    depth_marker = "        n = n_ = max(round(n * depth), 1) if n > 1 else n  # depth gain"
    if depth_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 depth 计算行，未执行修改")
    tasks_text = tasks_text.replace(depth_marker, depth_marker + "\n        m_ = None", 1)

    branch_marker = "        if m in base_modules:"
    branch = """        if m in edgevit_modules:
            if n != 1:
                raise ValueError("EdgeViT backbone only supports repeats=1")
            m_ = m(*args)
            c2 = m_.channel
        elif m in base_modules:
"""
    if branch_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 base_modules 分支，未执行修改")
    tasks_text = tasks_text.replace(branch_marker, branch, 1)

    instantiate = "        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    replacement = """        if m_ is None:
            m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"""
    if instantiate not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到模块实例化行，未执行修改")
    tasks_text = tasks_text.replace(instantiate, replacement, 1)

    tasks_backup = Path(str(tasks_path) + ".bak")
    if not tasks_backup.exists():
        shutil.copy2(tasks_path, tasks_backup)
    tasks_path.write_text(tasks_text, encoding="utf-8")
    print("已修改 tasks.py")
else:
    print("tasks.py 已经包含 EdgeViT parse_model 分支")

print("注册完成。请先执行下面的 YAML 单元，然后重启 Colab 运行时。")


modules/__init__.py 已经注册 EdgeViT
tasks.py 已经包含 EdgeViT parse_model 分支
注册完成。请先执行下面的 YAML 单元，然后重启 Colab 运行时。


In [8]:
# 创建 YOLO26 + EdgeViT-XXS 配置
# 数据集只有两个类别：0=UGV，1=USV。
# EdgeViT-XXS 输出 P3/P4/P5，通道为 [72, 144, 288]。
from pathlib import Path

cfg_path = Path(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"
)

cfg_path.write_text(
'''# YOLO26 with EdgeViT-XXS backbone.
# EdgeViT-XXS returns [P3, P4, P5] with channels [72, 144, 288].

nc: 2
end2end: True
reg_max: 1

backbone:
  - [-1, 1, EdgeViT_XXS, []]

head:
  - [0, 1, Index, [72, 0]]
  - [0, 1, Index, [144, 1]]
  - [0, 1, Index, [288, 2]]
  - [[1, 2, 3], 1, Detect, [nc]]
''',
    encoding="utf-8",
)

print("YAML:", cfg_path)
print(cfg_path.read_text(encoding="utf-8"))


YAML: /content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml
# YOLO26 with EdgeViT-XXS backbone.
# EdgeViT-XXS returns [P3, P4, P5] with channels [72, 144, 288].

nc: 2
end2end: True
reg_max: 1

backbone:
  - [-1, 1, EdgeViT_XXS, []]

head:
  - [0, 1, Index, [72, 0]]
  - [0, 1, Index, [144, 1]]
  - [0, 1, Index, [288, 2]]
  - [[1, 2, 3], 1, Detect, [nc]]



### 重启运行时后构建模型

执行上面的注册和 YAML 单元后，请选择：

运行时 → 重新启动运行时

重启后，从初始化单元重新执行到适配器单元，再执行下面的模型构建单元。这样可以清除旧的 ultralytics 和 tasks.py 导入缓存。


In [8]:
# 重启运行时后，构建 YOLO26 + EdgeViT 模型
%cd /content/ultralytics

from ultralytics import YOLO

DATA = "ul://yan-congjie/datasets/driverless-vehicle"
EDGEVIT_CFG = "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"

edge_model = YOLO(EDGEVIT_CFG)
edge_model.info(verbose=True)
print("EdgeViT model build passed")


/content/ultralytics
YOLO26_edgevit summary: 241 layers, 4,896,108 parameters, 4,896,108 gradients
EdgeViT model build passed


## 实验对比

In [ ]:
import os
os.environ['ULTRALYTICS_API_KEY'] = ''  # 替换成你的 key

## 实验 B：公平结构对比

本实验比较两个模型的结构差异：

- YOLO26n：从 YOLO26 YAML 随机初始化；
- YOLO26 + EdgeViT-XXS：从 EdgeViT YAML 随机初始化；
- 两个模型使用相同数据集、训练轮数、图像尺寸、batch、优化器、随机种子和验证/测试集；
- 两个模型都不加载预训练权重。

这样得到的是公平的结构对比。它与前面使用 yolo26n.pt 的实际应用对比不同，不能混用结论。


In [11]:
# 安装 Ultralytics Platform 支持和 asyncio 修复
# Jupyter/Colab 环境需要 nest_asyncio 来支持嵌套事件循环
%pip install -q nest_asyncio

import nest_asyncio
nest_asyncio.apply()

print("已启用嵌套事件循环支持")
print("Platform 云端数据集现在可以在 Colab 中使用")


已启用嵌套事件循环支持
Platform 云端数据集现在可以在 Colab 中使用


In [12]:
# 实验 B：统一配置并从 YAML 随机初始化两个模型
from pathlib import Path
import random
import numpy as np
import torch
from ultralytics import YOLO

DATA = "ul://yan-congjie/datasets/driverless-vehicle"
ULTRALYTICS_ROOT = Path("/content/ultralytics")
EDGEVIT_CFG = ULTRALYTICS_ROOT / "ultralytics/cfg/models/26/yolo26_edgevit.yaml"

# 优先使用明确的 yolo26n.yaml；如果当前 Ultralytics 版本只提供 yolo26.yaml，
# 则使用它并打印路径，便于确认当前版本的模型配置。
yolo26n_candidates = [
    ULTRALYTICS_ROOT / "ultralytics/cfg/models/26/yolo26n.yaml",
    ULTRALYTICS_ROOT / "ultralytics/cfg/models/26/yolo26.yaml",
]
YOLO26N_CFG = next((path for path in yolo26n_candidates if path.exists()), None)

assert DATA == "ul://yan-congjie/datasets/driverless-vehicle", "DATA 必须指向云端数据集"
assert EDGEVIT_CFG.exists(), f"EdgeViT 配置不存在: {EDGEVIT_CFG}"
assert YOLO26N_CFG is not None, "没有找到 yolo26n.yaml 或 yolo26.yaml"

SEED = 0
EPOCHS = 100
IMGSZ = 640
BATCH = 8
WORKERS = 2
DEVICE = 0

# 固定 Python、NumPy 和 PyTorch 随机种子。
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("DATA:", DATA)
print("YOLO26N_CFG:", YOLO26N_CFG)
print("EDGEVIT_CFG:", EDGEVIT_CFG)
print("seed:", SEED)
print("epochs:", EPOCHS)
print("imgsz:", IMGSZ)
print("batch:", BATCH)

# 两个模型均从 YAML 构建，因此这里没有加载 yolo26n.pt 或其他预训练权重。
torch.manual_seed(SEED)
baseline_scratch = YOLO(str(YOLO26N_CFG))

torch.manual_seed(SEED)
edge_scratch = YOLO(str(EDGEVIT_CFG))

print("\nYOLO26n architecture:")
baseline_scratch.info(verbose=True)

print("\nEdgeViT architecture:")
edge_scratch.info(verbose=True)


DATA: ul://yan-congjie/datasets/driverless-vehicle
YOLO26N_CFG: /content/ultralytics/ultralytics/cfg/models/26/yolo26.yaml
EDGEVIT_CFG: /content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml
seed: 0
epochs: 100
imgsz: 640
batch: 8
WARNING ⚠️ no model scale passed. Assuming scale='n'.

YOLO26n architecture:
YOLO26 summary: 260 layers, 2,572,280 parameters, 2,572,280 gradients, 6.2 GFLOPs

EdgeViT architecture:
YOLO26_edgevit summary: 241 layers, 4,896,108 parameters, 4,896,108 gradients


(241, 4896108, 4896108, 0.0)

In [19]:
# 使用完全相同的训练参数训练两个随机初始化模型
# 训练输出分别保存到两个独立目录，避免覆盖 checkpoint 和日志。
COMMON_TRAIN_ARGS = dict(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    deterministic=True,
    amp=True,
    patience=30,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    momentum=0.9,
    weight_decay=5e-4,
    project="/content/runs_platform_fair",
    exist_ok=False,
    pretrained=False,
)

print("Training scratch YOLO26n...")
baseline_scratch.train(
    **COMMON_TRAIN_ARGS,
    name="yolo26n_scratch",
)

print("Training scratch EdgeViT...")
edge_scratch.train(
    **COMMON_TRAIN_ARGS,
    name="edgevit_xxs_scratch",
)

print("Fair structure comparison training completed")


Training scratch YOLO26n...
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ul://yan-congjie/datasets/driverless-vehicle, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/ultralytics/ultralytics/cfg/models/26/yolo26.yaml, momentum=0.9, mo

/usr/local/lib/python3.13/dist-packages/simplejson/decoder.py:428: RuntimeWarning: coroutine 'convert_ndjson_to_yolo' was never awaited
  return self.scan_once(s, idx=_w(s, idx).end())


Platform: View model at https://platform.ultralytics.com/aday/runsplatformfair/yolo26nscratch-6
Converting driverless-vehicle.ndjson fraction=[1.0, 1.0, 0.0] → /content/datasets/driverless-vehicle-b2b3a104 using 620 train, 134 val, 0 test images: 100% ━━━━━━━━━━━━ 754/754 82.7it/s 9.1s<0.1s
Overriding model.yaml nc=80 with nc=2
WARNING ⚠️ no model scale passed. Assuming scale='n'.

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultraly

In [ ]:
# 加载两个最佳 checkpoint，并在同一个 test split 上评估
# 注意 在colan实验运行到本代码块时，需要重启运行时
# 同时把初始化到训练的所有代码块重新执行一遍（不包含训练代码块，不需要重启第二次）
from pathlib import Path
import json
from ultralytics import YOLO

baseline_best_path = Path(
    "/content/runs_platform_fair/yolo26n_scratch-6/weights/best.pt"
)
# 这里需要检查哪个yolo26scratch有权重
edge_best_path = Path(
    "/content/runs_platform_fair/edgevit_xxs_scratch/weights/best.pt"
)

assert baseline_best_path.exists(), baseline_best_path
assert edge_best_path.exists(), edge_best_path

baseline_best = YOLO(str(baseline_best_path))
edge_best = YOLO(str(edge_best_path))

baseline_metrics = baseline_best.val(
    data=DATA,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project="/content/eval_platform_fair",
    name="yolo26n_test",
)

edge_metrics = edge_best.val(
    data=DATA,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project="/content/eval_platform_fair",
    name="edgevit_xxs_test",
)

def metric_value(metrics, name):
    value = getattr(metrics.box, name, None)
    return None if value is None else float(value)

comparison = {
    "yolo26n_scratch": {
        "mAP50": metric_value(baseline_metrics, "map50"),
        "mAP50-95": metric_value(baseline_metrics, "map"),
        "precision": metric_value(baseline_metrics, "mp"),
        "recall": metric_value(baseline_metrics, "mr"),
        "parameters": sum(p.numel() for p in baseline_best.model.parameters()),
    },
    "edgevit_xxs_scratch": {
        "mAP50": metric_value(edge_metrics, "map50"),
        "mAP50-95": metric_value(edge_metrics, "map"),
        "precision": metric_value(edge_metrics, "mp"),
        "recall": metric_value(edge_metrics, "mr"),
        "parameters": sum(p.numel() for p in edge_best.model.parameters()),
    },
}

summary_path = Path("/content/eval_platform_fair/comparison.json")
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(comparison, indent=2, ensure_ascii=False))
print("saved:", summary_path)


Converting driverless-vehicle.ndjson fraction=1.0 → /content/datasets/driverless-vehicle-b0c1f8f9 using 620 train, 134 val, 0 test images: 100% ━━━━━━━━━━━━ 754/754 4.7Kit/s 0.2s0.2s
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26 summary (fused): 120 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs


FileNotFoundError: /content/datasets/driverless-vehicle-b0c1f8f9/data.yaml 'test:' images not found ❌

In [14]:
# Platform 数据集会自动下载到缓存目录
# 通常在 ~/.ultralytics/datasets/ 或 /content/datasets/
from pathlib import Path

# 首先尝试从 Platform 标准缓存位置读取
test_images = []

# Colab 上 Platform 缓存路径
platform_cache_paths = [
    Path("/content/datasets"),
    Path.home() / ".ultralytics" / "datasets",
]

for cache_root in platform_cache_paths:
    if not cache_root.exists():
        continue
    # 查找包含 test 子目录的数据集
    for dataset_dir in cache_root.iterdir():
        if not dataset_dir.is_dir():
            continue
        test_dir = dataset_dir / "test"
        if test_dir.exists():
            for img_path in test_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                    test_images.append(str(img_path))
        if test_images:
            print(f"Found test images in: {dataset_dir}")
            break
    if test_images:
        break

if not test_images:
    print("未找到 Platform 缓存的测试图片")
    print("训练完成后会自动缓存数据集，请重新运行此单元")
    print("或手动指定测试图片路径")
else:
    test_images = sorted(test_images)
    print("test images:", len(test_images))



未找到 Platform 缓存的测试图片
训练完成后会自动缓存数据集，请重新运行此单元
或手动指定测试图片路径


In [15]:
# 纯模型前向延迟对比，不包含 Ultralytics 的图像预处理和 NMS
import time
import torch

def benchmark_core(model, repeats=30, warmup=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = model.model.to(device).eval()
    dummy = torch.randn(1, 3, IMGSZ, IMGSZ, device=device)

    with torch.inference_mode():
        for _ in range(warmup):
            _ = net(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(repeats):
            _ = net(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

    latency_ms = (time.perf_counter() - start) * 1000 / repeats
    return latency_ms

baseline_latency = benchmark_core(baseline_best)
edge_latency = benchmark_core(edge_best)

comparison["yolo26n_scratch"]["core_latency_ms"] = baseline_latency
comparison["edgevit_xxs_scratch"]["core_latency_ms"] = edge_latency

summary_path.write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("YOLO26n core latency (ms):", round(baseline_latency, 3))
print("EdgeViT core latency (ms):", round(edge_latency, 3))
print("updated:", summary_path)


NameError: name 'comparison' is not defined

## 实验 B 结果解释

结果应同时查看：

- mAP50 和 mAP50-95；
- UGV、USV 两个类别的 AP；
- Precision 和 Recall；
- 参数量；
- 纯模型延迟；
- predict_platform_fair 下的同图预测结果。

由于两个模型都从随机初始化开始，结果可以用于结构对比。不要把这组结果与 yolo26n.pt 预训练模型的结果直接当作同一实验结论。
